## Install Stylish-TTS and k2

In [ ]:
%cd /content
# Install pytorch and onnx.
# Use onnxruntime-gpu if you want to do test inference with a GPU.
# k2 have not supported torch 2.10+
!uv pip install "torch<2.10" "torchaudio<2.10" onnxruntime --system

# Install k2 for Linux or WSL.
!wget "https://github.com/Fannovel16/stylish-tts/raw/refs/heads/fix-main-bugs/get_k2_whl.py" && uv pip install "k2 @ $(uv run get_k2_whl.py)" --system

# Verify that k2 was installed successfully
!uv run python -c "import k2; print('k2 installed successfully')"

# Clone the stylish-tts source somewhere (TODO: Fix this when we upload a package)
!git clone https://github.com/Stylish-TTS/stylish-tts.git

# Install stylish-tts as a local editable package
# Automatically rebuilds if contents change
!uv pip install --editable stylish-tts/ --system

## Download LJSpeech denoised by Sidon

In [ ]:
%cd /content
!sudo apt install aria2 -y
!rm ljspeech-sidon-24khz.zip
!aria2c -x 16 -o ljspeech-sidon-24khz.zip https://huggingface.co/datasets/hr16/ljspeech-sidon/resolve/main/ljspeech-sidon-24khz.zip
print("Unzipping...")
!unzip -qq ljspeech-sidon-24khz.zip
print("Finished")
!mkdir ckpt

## Write config files

In [ ]:
%%writefile /content/ckpt/config.yml
training:
  # log training data every this number of steps
  log_interval: 1000
  # validate and save model every this number of steps
  save_interval: 5000
  # validate model every this number of steps
  val_interval: 5000
  device: "cuda"
  # Keep this as 'no' if you have the VRAM.
  # Lower precision slows training.
  # "bf16", "fp16", or "no" for no mixed precision
  mixed_precision: "no"
  # Reserve vRAM buffer during probe to avoid OOMs
  vram_reserve: 200
  data_workers: 0

# Number of epochs, max batch sizes and general learning rate of each stage.
training_plan:
  alignment:
    # alignment pretraining
    epochs: 10
    # Maximum number of segments per batch.
    probe_batch_max: 16
    # Learing Rate for this stage
    lr: 1e-5
  acoustic:
    # training of acoustic models and vocoder
    epochs: 10
    probe_batch_max: 8
    lr: 1e-4
  textual:
    # training for duration/pitch/energy from text
    epochs: 20
    probe_batch_max: 32
    lr: 3e-5
  style:
    # training for style from text
    epochs: 10
    probe_batch_max: 64
    lr: 1e-5
  joint:
    # Experimental joint training
    epochs: 10
    probe_batch_max: 16
    lr: 1e-5
  duration:
    epochs: 20
    probe_batch_max: 32
    lr: 1e-4

dataset:
  # All paths in this section are relative to the main path
  path: "/content/ljspeech-sidon-24khz"
  train_data: "train-list.txt"
  val_data: "val-list.txt"
  wav_path: "wavs"
  pitch_path: "pitch.safetensors"
  alignment_path: "alignment.safetensors"
  alignment_model_path: "alignment_model.safetensors"

validation:
  # Number of samples to generate per validation step
  sample_count: 10
  # Specific segments to use for validation
  # force_samples:
  # - "filename.from.val_data.txt"
  # - "other.filename.from.val_data.txt"
  # - "other.other.filename.from.val_data.txt"


  # Weights are pre-tuned. Do not change these unless you
  # know what you are doing.
loss_weight:
  # mel reconstruction loss
  mel: 5
  # generator loss
  # -- if voice sounds autotuned, try increasing
  generator: 1
  # speech-language model feature matching loss
  slm: 0.2
  # pitch F0 reconstruction loss
  pitch: 8
  # energy reconstruction loss
  energy: 8
  # duration loss
  # -- if voice is too slow and pauses are too long, try increasing this and decreasing the other
  duration: 8
  # duration predictor probability output cross entropy loss
  # -- if voice is too fast and pauses are too short, try increasing this and decreasing the other
  duration_ce: 8
  # style reconstruction loss
  style: 1
  # magnitude loss
  mag: 1
  # phase loss
  # -- if voice sounds autotuned, try increasing
  phase: 8
  # Loss for predicting which parts of the segment are voiced/unvoiced
  voiced: 1
  # multi-phase loss
  multi_phase: 8
  # confidence for alignment (placeholder)
  confidence: 1
  # alignment loss
  align_loss: 1
  # discriminator loss (placeholder)
  discriminator: 1

In [ ]:
%%writefile /content/vctk_processed/model.yml
# Configuration for training on a high-resource (24GB+ VRAM) GPU.

multispeaker: false
sample_rate: 24000
n_mels: 80
n_fft: 512
win_length: 512
hop_length: 300
coarse_multiplier: 1
style_dim: 64
inter_dim: 128

text_aligner:
  n_mels: 80
  n_fft: 2048
  win_length: 1200
  hop_length: 300
  hidden_dim: 256
  token_embedding_dim: 512

decoder:
  hidden_dim: 128
  residual_dim: 64

# generator:
#   type: 'ringformer'
#   resblock_kernel_sizes: [ 3, 7, 11 ]
#   upsample_rates: [ 5 ]
#   input_dim: 256
#   upsample_initial_channel: 256
#   upsample_last_channel: 128
#   resblock_dilation_sizes: [ [ 1, 3, 5 ], [ 1, 3, 5 ], [ 1, 3, 5 ] ]
#   upsample_kernel_sizes: [ 10 ]
#   gen_istft_n_fft: 60
#   gen_istft_hop_size: 15
#   depth: 2

generator:
  type: 'freegan'
  input_dim: 128
  hidden_dim: 256
  conv_intermediate_dim: 768
  io_conv_kernel_size: 21
  conformer_layers: 1
  conv_layers: 8

text_encoder:
  tokens: 178 # number of phoneme tokens
  hidden_dim: 128
  filter_channels: 512
  heads: 8
  layers: 8
  kernel_size: 3
  dropout: 0.2

style_encoder:
  n_mels: 80
  n_fft: 2048
  win_length: 1200
  hop_length: 300
  max_channels: 384
  skip_downsample: True

duration_predictor:
  n_layer: 3
  duration_classes: 16
  max_duration: 50
  dropout: 0.5
  last_dropout: 0.5

pitch_energy_predictor:
  inter_dim: 256
  dropout: 0.2

# speech language model config
slm:
  model: 'microsoft/wavlm-base-plus'
  sr: 16000 # sampling rate of SLM

symbol:
  pad: "$"
  punctuation: ";:,.!?¡¿—…\"()“” "
  letters: "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
  letters_ipa: "ɑɐɒæɓʙβɔɕçɗɖðʤəɘɚɛɜɝɞɟʄɡɠɢʛɦɧħɥʜɨɪʝɭɬɫɮʟɱɯɰŋɳɲɴøɵɸθœɶʘɹɺɾɻʀʁɽʂʃʈʧʉʊʋⱱʌɣɤʍχʎʏʑʐʒʔʡʕʢǀǁᵊǃˈˌːˑʼʴʰʱʲʷˠˤ˞↓↑→↗↘'̩'ᵻ"

## Dataset Preparation

### Prepare pitch

In [ ]:
%cd /content
!uv run stylish-train pitch ckpt/config.yml -mc ckpt/model.yml --method rmvpe --workers 8

### Train aligner (negative loss is normal)

In [ ]:
%cd /content
!uv run stylish-train train-align ckpt/config.yml -mc ckpt/model.yml --out ckpt

### Prepare alignment

In [ ]:
%cd /content
!uv run stylish-train align ckpt/config.yml -mc ckpt/model.yml --method k2

In [ ]:
!rm -r /content/ckpt/acoustic

## Begin Training

In [ ]:
%cd /content
!uv run stylish-train train ckpt/config.yml -mc ckpt/model.yml --out ckpt